In [1]:
import requests
import PyPDF2
from io import BytesIO
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import requests
import httpx

In [32]:
dotenv_path = "../.env"
dotenv_path = Path("../.env")
load_dotenv(dotenv_path=dotenv_path)

gemini_api_key = os.getenv("GEMINI_API_KEY2")

In [33]:
from google import genai
from google.genai import types

client = genai.Client(api_key=gemini_api_key)

In [4]:
Statementdirectory = "./Statements/"
Promptdirectory = "./Prompts/"

## Load Data

In [5]:
GraderchanDF = pd.read_csv(Statementdirectory+"graderchan_clean.csv")
ProgramminDF = pd.read_csv(Statementdirectory+"programmingin_clean.csv")

## Load Prompt

In [34]:
with open(Promptdirectory+'PromptB0.txt','r') as cp:
  promptB0 = cp.read()

with open(Promptdirectory+'PromptB1.txt','r') as cp:
  promptB1 = cp.read()  
  
with open(Promptdirectory+'PromptV1.txt','r') as cp:
  promptV1 = cp.read()
  
with open(Promptdirectory+'PromptV2.txt','r') as cp:
  promptV2 = cp.read()
  
with open(Promptdirectory+'PromptV3.txt','r') as cp:
  promptV3 = cp.read()
  

In [30]:
promptB1

'You are a programmer. Your task is to solve the problem below using C++.\nanswer your answer with ```cpp {code} ``` format and do not provide other text except code.\nYou could write comment in your code.'

## Functions

In [35]:
SolutionDF = pd.DataFrame(columns=['writer', 'problem_id', 'submission_id', 'website', 'source'])

In [36]:
def pdfURL_to_text(url):
    res = requests.get(url)
    with open("./Temps/temp.pdf", "wb") as f:
        f.write(res.content)
    pdf_reader = PyPDF2.PdfReader(BytesIO(res.content))
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()
    print(text)
    return text
    

In [37]:
def SolutionInsert(source,website,problem_id):
    global SolutionDF
    row = {
        'writer':'AI',
        'problem_id':problem_id,
        'submission_id':np.nan,
        'website':website,
        'source':source
    }
    print(row)
    SolutionDF = pd.concat([SolutionDF,pd.DataFrame([row])],ignore_index=True)
    return row

## Gemini 2.0 Flash (PDF)

In [38]:
def GeminiGenerateSol(url,prompt_text=promptV1,model="gemini-2.0-flash"):
    try:
        doc_data = httpx.get(url).content
        response = client.models.generate_content(
        model=model,
        contents=[
            types.Part.from_bytes(
                data=doc_data,
                mime_type='application/pdf',
            ),
            prompt_text])
        try :
            answer = response.text.removeprefix("```cpp").removesuffix("```").strip()
        except Exception as e:
            return None, e
    except Exception as e:
        return None, e
    return answer, response

### GraderChan

In [ ]:
SolutionDF = pd.DataFrame(columns=['writer', 'problem_id', 'submission_id', 'website', 'source'])
from IPython.display import clear_output
import time

SAVE_PATH = './Generated_Solution/Gemini_Graderchan_solV3-1.csv'

# Gchan Gen
for index, row in GraderchanDF.iterrows():
    error_cnt = 0
    while True:
        if error_cnt >= 2:
            ans = None
        ans, res = GeminiGenerateSol(row['ProblemURL'],promptV3)

        if ans is not None:
            break  # Exit loop if answer is good
        else:
            error_message = str(res)
            if ('The document has no pages.' in error_message):
                print(f"Statement : {row['PdfURL']} not found")
                break
            print(row['PdfURL'])
            print(f"ERROR : {error_message}")
            print(f"Failed at index {index}, retrying in 30 seconds...")
            SolutionDF.to_csv(SAVE_PATH,index=False)
            time.sleep(30)  # Wait before retry
            error_cnt+=1

    if ans == None:
        continue
    clear_output()
    print(index, "\n", ans)
    SolutionInsert(ans, 'https://firefly.gchan.moe', row['problemID'])
    if index%50 == 0:
        SolutionDF.to_csv(SAVE_PATH,index=False)
SolutionDF.to_csv(SAVE_PATH,index=False)

180 
 #include <bits/stdc++.h>
using namespace std;
int main() {
  int n, k;
  cin >> n >> k;
  vector<int> a(n);
  vector<int> b(n);
  for (int i = 0; i < n; i++) {
    cin >> a[i];
  }
  for (int i = 0; i < n; i++) {
    cin >> b[i];
  }
  int days = 0;
  while (true) {
    int total_zombies = 0;
    for (int i = 0; i < n; i++) {
      total_zombies += a[i];
      total_zombies += b[i];
    }
    if (total_zombies == 0) {
      cout << "YAY\n";
      return 0;
    }
    days++;
    int bullets = k;
    for (int i = 0; i < n; i++) {
      int reduce_a = min(a[i], bullets);
      a[i] -= reduce_a;
      bullets -= reduce_a;
      int reduce_b = min(b[i], bullets);
      b[i] -= reduce_b;
      bullets -= reduce_b;
    }
    if (a[0] != 0) {
      cout << "GG\n";
      return 0;
    }
    if (b[0] != 0) {
      cout << "GG\n";
      return 0;
    }
    for (int i = 0; i < n - 1; i++) {
      a[i] = a[i + 1];
      b[i] = b[i + 1];
    }
    a[n - 1] = 0;
    b[n - 1] = 0;
  }
  return 0

### ProgrammingIN

In [42]:
# SolutionDF = pd.DataFrame(columns=['writer', 'problem_id', 'submission_id', 'website', 'source'])
from IPython.display import clear_output
import time
SAVE_PATH = './Generated_Solution/Gemini_Programming_solB1-1.csv'

for index, row in ProgramminDF.iterrows():
    if index < 42:
        continue
    error_cnt = 0
    while True:
        if error_cnt >= 2:
            ans = None
            break
        ans, res = GeminiGenerateSol(row['PdfURL'],promptB1)
        if ans is not None:
            break
        else:
            print(res)
        if not ans == None:  
                break
        else:  
            error_message = str(res)
            if ('The document has no pages.' in error_message):
                print(f"Statement : {row['PdfURL']} not found")
                break
            print(row['PdfURL'])
            print(f"ERROR : {error_message}")
            print(f"Failed at index {index}, retrying in 30 seconds...")
            SolutionDF.to_csv(SAVE_PATH,index=False)
            time.sleep(30)
            error_cnt+=1
    if ans == None:
        continue
    clear_output()
    print(index, "\n", ans)
    SolutionInsert(ans, 'https://programming.in.th', row['ProblemID'])
    if index%50 == 0:
        SolutionDF.to_csv(SAVE_PATH,index=False)
SolutionDF.to_csv(SAVE_PATH,index=False)
    

244 
 #include <iostream>
#include <vector>
#include <queue>
#include <algorithm>

using namespace std;

int main() {
    int n, m, k;
    cin >> n >> m >> k;

    vector<int> v(n + 1);
    for (int i = 1; i <= n; ++i) {
        cin >> v[i];
    }

    vector<vector<int>> adj(n + 1);
    for (int i = 0; i < m; ++i) {
        int u, w;
        cin >> u >> w;
        adj[u].push_back(w);
        adj[w].push_back(u);
    }

    long long max_sum = -1e18; // Initialize with a very small value

    for (int i = 0; i < (1 << n); ++i) { // Iterate through all possible subsets
        long long current_sum = 0;
        vector<bool> taken(n + 1, false);
        vector<bool> infected(n + 1, false);

        // Mark the initially infected node
        infected[k] = true;

        // Nodes we take out before the virus spreads.
        vector<int> taken_nodes;

        // Create the subset of taken nodes based on the bitmask
        for (int j = 0; j < n; ++j) {
            if ((i >> j) & 1) {
    

KeyboardInterrupt: 

### Save

In [43]:
SolutionDF

,writer,problem_id,submission_id,website,source
0,AI,0011,NaN,https://programming.in.th,#include <iostream>\n#include <vector>\n#inclu...
1,AI,0012,NaN,https://programming.in.th,#include <iostream>\n#include <string>\n#inclu...
2,AI,0013,NaN,https://programming.in.th,#include <iostream>\n#include <vector>\n#inclu...
3,AI,0014,NaN,https://programming.in.th,#include <iostream>\n\nusing namespace std;\n\...
4,AI,0015,NaN,https://programming.in.th,#include <iostream>\n#include <algorithm>\n\nu...
...,...,...,...,...,...
212,AI,codecube_019,NaN,https://programming.in.th,#include <iostream>\n#include <vector>\n#inclu...
213,AI,codecube_021,NaN,https://programming.in.th,#include <iostream>\n#include <vector>\n\nusin...
214,AI,codecube_022,NaN,https://programming.in.th,#include <iostream>\n#include <set>\n\nusing n...
215,AI,codecube_026,NaN,https://programming.in.th,#include <iostream>\n#include <vector>\n#inclu...


In [24]:
SolutionDF = SolutionDF[SolutionDF['website']=='https://firefly.gchan.moe']

In [44]:
SolutionDF.to_csv(SAVE_PATH,index=False)